In [1]:
# 1. Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt

# 2. Load Dataset
df = pd.read_csv("high_diamond_ranked_10min.csv")

# 3. Feature Engineering
df['kda'] = (df['blueKills'] + df['blueAssists']) / (df['blueDeaths'] + 1)
df['vision_score'] = df['blueWardsPlaced'] + df['blueWardsDestroyed']
df['aggression'] = df['blueKills'] / (df['blueDeaths'] + 1)
df['teamplay'] = df['blueAssists'] / (df['blueKills'] + 1)

# Normalize behavior features
scaler = MinMaxScaler()
behavior_features = ['kda', 'vision_score', 'aggression', 'teamplay']
df[behavior_features] = scaler.fit_transform(df[behavior_features])

# Behavior + Performance score
df['behavior_score'] = (
    0.35 * df['kda'] +
    0.25 * df['teamplay'] +
    0.25 * df['vision_score'] +
    0.15 * df['aggression']
)

df['performance_score'] = (
    df['blueKills'] +
    0.7 * df['blueAssists'] -
    df['blueDeaths']
)

# 4. Classification Labels
def classify_player(row):
    if row['performance_score'] > 8 and row['behavior_score'] > 0.6:
        return 0
    elif row['performance_score'] <= 8 and row['behavior_score'] > 0.6:
        return 1
    elif row['performance_score'] <= 8 and row['behavior_score'] <= 0.6:
        return 2
    else:
        return 3

df['player_type'] = df.apply(classify_player, axis=1)

# 5. Model Data
X = df[['blueKills','blueDeaths','blueAssists','blueGoldDiff',
        'blueExperienceDiff','kda','vision_score','teamplay','aggression']]

y = df['player_type']

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 7. Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 8. Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("LR Accuracy:", accuracy_score(y_test, y_pred_lr))

# 9. Random Forest
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))

# 10. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)

plt.imshow(cm)
plt.title("Confusion Matrix")
for i in range(len(cm)):
    for j in range(len(cm)):
        plt.text(j, i, cm[i][j], ha='center')
plt.show()

<class 'FileNotFoundError'>: [Errno 44] No such file or directory: 'high_diamond_ranked_10min.csv'